In [1]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [2]:
!pip install jpeg4py
!pip install lmdb
!pip install torchinfo
!pip install visdom
!pip install spikingjelly

  Preparing metadata (setup.py) ... done
  Created wheel for jpeg4py: filename=jpeg4py-0.1.4-py3-none-any.whl size=8426 sha256=0d5b8e1f8b4871b9ebce8c7788f9cbbd0a51dd52776d51fe00d2aabffd045972
  Stored in directory: /root/.cache/pip/wheels/ab/a3/c5/3fce1e4f84180fa8362da30c3db4c719be956d49a0da49bd47
Successfully built jpeg4py
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.1/333.1 kB 11.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 18.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for visdom: filename=visdom-0.2.4-py3-none-any.whl size=1408195 sha256=8dfe633468c761a10284b983500e8ef7a0f6a8de66216c66cdafb42940818303
  Stored in directory: /root/.cache/pip/wheels/37/6c/38/64eeaa310e325aacda723e6df1f79ab5e9f31ba195264e04a8
Successfully built visdom
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 437.6/437.6 kB 14.9 MB/s eta 0:00:00


In [3]:
import shutil
import os
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

In [4]:
!git clone https://github.com/Event-AHU/EventVOT_Benchmark.git
!cd EventVOT_Benchmark/HDETrack

Cloning into 'EventVOT_Benchmark'...
remote: Enumerating objects: 1096, done.
remote: Counting objects: 100% (126/126), done.
remote: Compressing objects: 100% (95/95), done.
remote: Total 1096 (delta 80), reused 31 (delta 31), pack-reused 970 (from 3)
Receiving objects: 100% (1096/1096), 42.58 MiB | 18.03 MiB/s, done.
Resolving deltas: 100% (509/509), done.


In [5]:
file_path = '/content/EventVOT_Benchmark/HDETrack/lib/train/data/loader.py'

with open(file_path, 'r') as f:
    content = f.read()

old = """import torch
import torch.utils.data.dataloader
import importlib
import collections
from torch._six import string_classes
from lib.utils import TensorDict, TensorList

if float(torch.__version__[:3]) >= 1.9 or len('.'.join((torch.__version__).split('.')[0:2])) > 3:
    int_classes = int
else:
    from torch._six import int_classes"""

new = """import torch
import torch.utils.data.dataloader
import importlib
import collections
#from torch._six import string_classes
from lib.utils import TensorDict, TensorList
if float(torch.__version__[:3]) >= 1.9 or len('.'.join((torch.__version__).split('.')[0:2])) > 3:
    int_classes = int
else:
    from torch._six import int_classes
if float(torch.__version__[:3]) >= 1.9 or len('.'.join((torch.__version__).split('.')[0:2])) > 3:
    string_classes = str
else:
    from torch._six import string_classes"""

content = content.replace(old, new)

with open(file_path, 'w') as f:
    f.write(content)

print("Done!")

Done!


In [6]:
file_path = '/content/EventVOT_Benchmark/HDETrack/lib/test/tracker/hdetrack.py'

with open(file_path, 'r') as f:
    content = f.read()

content = content.replace(
    "network.load_state_dict(torch.load(self.params.checkpoint, map_location='cpu')['net'], strict=True)",
    "network.load_state_dict(torch.load(self.params.checkpoint, map_location='cpu', weights_only=False)['net'], strict=True)"
)

with open(file_path, 'w') as f:
    f.write(content)

print("Done!")

Done!


In [7]:
%cd /content/EventVOT_Benchmark/HDETrack

!python tracking/create_default_local_file.py \
    --workspace_dir . \
    --data_dir ./data \
    --save_dir ./output

/content/EventVOT_Benchmark/HDETrack
2026-04-27 17:34:38.438306: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777311278.461032    2469 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777311278.468509    2469 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777311278.487740    2469 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777311278.487796    2469 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777311278.487802    2469 computation_

In [8]:
# Source file paths
ceu = '/content/drive/MyDrive/Diploma_project/HDETrack/CEUTrack_ep0050.pth.tar'
hde = '/content/drive/MyDrive/Diploma_project/HDETrack/HDETrack_S_ep0050.pth.tar'
vit = '/content/drive/MyDrive/Diploma_project/HDETrack/mae_pretrain_vit_base.pth'

# Target directories as required by the docs
pretrained_dir = '/content/EventVOT_Benchmark/HDETrack/pretrained_models'
checkpoint_dir = '/content/EventVOT_Benchmark/HDETrack/output/checkpoints/train/hdetrack/hdetrack_eventvot'

# Create directories if they don't exist
os.makedirs(pretrained_dir, exist_ok=True)
os.makedirs(checkpoint_dir, exist_ok=True)

# MAE ViT-Base → pretrained_models/
shutil.copy(vit, os.path.join(pretrained_dir, 'mae_pretrain_vit_base.pth'))

# CEUTrack (teacher model) → pretrained_models/
shutil.copy(ceu, os.path.join(pretrained_dir, 'CEUTrack_ep0050.pth.tar'))

# HDETrack_S (student model, for inference) → output/checkpoints/.../
shutil.copy(hde, os.path.join(checkpoint_dir, 'HDETrack_S_ep0050.pth.tar'))

# Verify the result
print("\nFinal file structure:")
for path in [pretrained_dir, checkpoint_dir]:
    for root, dirs, files in os.walk(path):
        for f in files:
            print(f"  {os.path.join(root, f)}")


Final file structure:
  /content/EventVOT_Benchmark/HDETrack/pretrained_models/mae_pretrain_vit_base.pth
  /content/EventVOT_Benchmark/HDETrack/pretrained_models/CEUTrack_ep0050.pth.tar
  /content/EventVOT_Benchmark/HDETrack/output/checkpoints/train/hdetrack/hdetrack_eventvot/HDETrack_S_ep0050.pth.tar


In [21]:
sequences = sorted(os.listdir('/content/drive/MyDrive/Diploma_project/Datasets/FE108(v2)/test'))
sequences

['airplane_mul222',
 'bike222',
 'bike333',
 'bike_low',
 'bottle_mul222',
 'box_hdr',
 'box_low',
 'cow_mul222',
 'cup222',
 'cup_low',
 'dog',
 'dog_motion',
 'dove_motion',
 'fighter_mul',
 'giraffe222',
 'giraffe_low',
 'giraffe_motion',
 'ship',
 'ship_motion',
 'star',
 'star_mul',
 'star_mul222',
 'tank_low',
 'tower',
 'tower333',
 'truck',
 'truck_hdr',
 'whale_mul222']

In [29]:
!rm -rf '/content/EventVOT_Benchmark/HDETrack/data'

!rm -rf '/content/EventVOT_Benchmark/HDETrack/output/test'

In [23]:
src_root = '/content/drive/MyDrive/Diploma_project/Datasets/FE108(v2)/test'
base = '/content/EventVOT_Benchmark/HDETrack/data/EventVOT/test'

sequences = sorted(os.listdir(src_root))

# Copy each sequence
for seq_name in sequences:
    dvs = os.path.join(src_root, seq_name, 'dvs')
    gt = os.path.join(src_root, seq_name, 'groundtruth_rect.txt')
    img_dst = os.path.join(base, seq_name, 'img')

    os.makedirs(img_dst, exist_ok=True)

    # Copy frames
    frames = [f for f in sorted(os.listdir(dvs)) if f.endswith('.png') or f.endswith('.bmp')]
    for fname in frames:
        shutil.copy(os.path.join(dvs, fname), os.path.join(img_dst, fname))

    # Copy groundtruth
    shutil.copy(gt, os.path.join(base, seq_name, 'groundtruth.txt'))

    print(f"{seq_name}: {len(frames)} frames copied")

print("\nDone!")

airplane_mul222: 2051 frames copied
bike222: 1899 frames copied
bike333: 2001 frames copied
bike_low: 1290 frames copied
bottle_mul222: 1101 frames copied
box_hdr: 1948 frames copied
box_low: 2084 frames copied
cow_mul222: 2231 frames copied
cup222: 2010 frames copied
cup_low: 1933 frames copied
dog: 642 frames copied
dog_motion: 2788 frames copied
dove_motion: 2202 frames copied
fighter_mul: 2000 frames copied
giraffe222: 2390 frames copied
giraffe_low: 2267 frames copied
giraffe_motion: 1500 frames copied
ship: 967 frames copied
ship_motion: 2301 frames copied
star: 1156 frames copied
star_mul: 2160 frames copied
star_mul222: 2036 frames copied
tank_low: 2276 frames copied
tower: 1144 frames copied
tower333: 2401 frames copied
truck: 1131 frames copied
truck_hdr: 1969 frames copied
whale_mul222: 2171 frames copied

Done!


In [24]:
src_root = '/content/drive/MyDrive/Diploma_project/Datasets/FE108(v2)/test'
base = '/content/EventVOT_Benchmark/HDETrack/data/EventVOT/test'

sequences = sorted(os.listdir(src_root))

os.makedirs(base, exist_ok=True)

with open(os.path.join(base, 'list.txt'), 'w') as f:
    for seq_name in sequences:
        f.write(seq_name + '\n')

print(f"list.txt created with {len(sequences)} sequences")

list.txt created with 28 sequences


In [25]:
!python tracking/test.py hdetrack hdetrack_eventvot --dataset eventvot --threads 1 --num_gpus 1

2026-04-26 20:31:10.342688: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777235470.377587   55934 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777235470.388845   55934 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777235470.422394   55934 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777235470.422426   55934 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777235470.422432   55934 computation_placer.cc:177] computation placer alr

In [26]:
def calc_metrics(gt, pr):
        min_len = min(len(gt), len(pr))
        gt = gt[:min_len]
        pr = pr[:min_len]

        gt_centers = gt[:, :2] + gt[:, 2:4] / 2
        pr_centers = pr[:, :2] + pr[:, 2:4] / 2
        distances  = np.sqrt(np.sum((gt_centers - pr_centers) ** 2, axis=1))
        precision  = np.mean(distances <= 20)

        ix1 = np.maximum(gt[:, 0], pr[:, 0])
        iy1 = np.maximum(gt[:, 1], pr[:, 1])
        ix2 = np.minimum(gt[:, 0] + gt[:, 2], pr[:, 0] + pr[:, 2])
        iy2 = np.minimum(gt[:, 1] + gt[:, 3], pr[:, 1] + pr[:, 3])

        inter_area = np.maximum(0, ix2 - ix1) * np.maximum(0, iy2 - iy1)
        union_area = gt[:, 2] * gt[:, 3] + pr[:, 2] * pr[:, 3] - inter_area
        iou        = inter_area / (union_area + 1e-10)
        success    = np.mean(iou > 0.5)

        return precision, success

In [28]:
rows = []
for seq in sequences:
    gt = f'/content/EventVOT_Benchmark/HDETrack/data/EventVOT/test/{seq}/groundtruth.txt'
    pr = f'/content/EventVOT_Benchmark/HDETrack/output/test/tracking_results/hdetrack/hdetrack_eventvot/{seq}.txt'

    gt_data = np.loadtxt(gt, delimiter=',')
    pr_data = np.loadtxt(pr)
    pr, sr = calc_metrics(gt_data, pr_data)

    rows.append({
        "sequence":     seq,
        "PR (HDE)":     pr,
        "SR@0.5 (HDE)": sr,
    })

df = pd.DataFrame(rows)

# Print sequences with 2 decimals
print(df.to_string(index=False, float_format=lambda x: f"{x:.2f}"))

# Print mean separately with 3 decimals
print("-" * 40)
print(f"{'MEAN':<25} {df['PR (HDE)'].mean():.3f}   {df['SR@0.5 (HDE)'].mean():.3f}")

       sequence  PR (HDE)  SR@0.5 (HDE)
airplane_mul222      0.86          0.67
        bike222      0.14          0.10
        bike333      0.13          0.08
       bike_low      1.00          0.99
  bottle_mul222      0.24          0.21
        box_hdr      0.94          0.88
        box_low      0.99          0.99
     cow_mul222      0.67          0.65
         cup222      0.21          0.13
        cup_low      0.93          0.92
            dog      1.00          0.86
     dog_motion      0.51          0.42
    dove_motion      0.58          0.53
    fighter_mul      0.19          0.08
     giraffe222      0.07          0.01
    giraffe_low      0.93          0.66
 giraffe_motion      0.49          0.28
           ship      1.00          0.42
    ship_motion      0.07          0.02
           star      1.00          0.24
       star_mul      0.38          0.22
    star_mul222      0.85          0.71
       tank_low      1.00          1.00
          tower      0.99          0.90


## Illumination dataset

In [9]:
initials = 'seya'
#sequences = ['ball_1:2hz', 'ball_1hz', 'ball_2hz', 'banana_1:2hz', 'banana_1hz', 'banana_2hz', 'd3_1:2hz', 'd3_1hz', 'd3_2hz']

sequences = [
    s for s in sorted(os.listdir('/content/drive/MyDrive/Diploma_project/Datasets/Illumination_custom/seya/flicker'))[1:]
    if not s.endswith('.zip')
    and not s.startswith('pen')
    and not s.startswith('spinner')
    and not s.endswith('4hz')
    and not s.endswith('8hz')
]

sequences

['ball_1:2hz',
 'ball_1hz',
 'ball_2hz',
 'banana_1:2hz',
 'banana_1hz',
 'banana_2hz',
 'cleaner_1:2hz',
 'cleaner_1hz',
 'cleaner_2hz',
 'd3_1:2hz',
 'd3_1hz',
 'd3_2hz',
 'key_1:2hz',
 'key_1hz',
 'key_2hz',
 'knife_1:2hz',
 'knife_1hz',
 'knife_2hz',
 'lamp_1:2hz',
 'lamp_1hz',
 'lamp_2hz',
 'longus_1:2hz',
 'longus_1hz',
 'longus_2hz',
 'nonstop_1:2hz',
 'nonstop_1hz',
 'nonstop_2hz',
 'repeater_1:2hz',
 'repeater_1hz',
 'repeater_2hz',
 'rexona_1:2hz',
 'rexona_1hz',
 'rexona_2hz',
 'soap_1:2hz',
 'soap_1hz',
 'soap_2hz',
 'white_1:2hz',
 'white_1hz',
 'white_2hz']

In [10]:
src_root = '/content/drive/MyDrive/Diploma_project/Datasets/Illumination_custom/seya/flicker'
base = '/content/EventVOT_Benchmark/HDETrack/data/EventVOT/test'


# Copy each sequence
for seq_name in sequences:
    dvs = os.path.join(src_root, seq_name, 'dvs')
    gt = os.path.join(src_root, seq_name, 'groundtruth_rect.txt')
    img_dst = os.path.join(base, seq_name, 'img')

    os.makedirs(img_dst, exist_ok=True)

    # Copy frames
    frames = [f for f in sorted(os.listdir(dvs)) if f.endswith('.png') or f.endswith('.bmp')]
    for fname in frames:
        shutil.copy(os.path.join(dvs, fname), os.path.join(img_dst, fname))

    # Copy groundtruth
    shutil.copy(gt, os.path.join(base, seq_name, 'groundtruth.txt'))

    print(f"{seq_name}: {len(frames)} frames copied")

print("\nDone!")

ball_1:2hz: 214 frames copied
ball_1hz: 202 frames copied
ball_2hz: 204 frames copied
banana_1:2hz: 246 frames copied
banana_1hz: 226 frames copied
banana_2hz: 213 frames copied
cleaner_1:2hz: 206 frames copied
cleaner_1hz: 207 frames copied
cleaner_2hz: 191 frames copied
d3_1:2hz: 209 frames copied
d3_1hz: 215 frames copied
d3_2hz: 209 frames copied
key_1:2hz: 217 frames copied
key_1hz: 221 frames copied
key_2hz: 207 frames copied
knife_1:2hz: 191 frames copied
knife_1hz: 205 frames copied
knife_2hz: 209 frames copied
lamp_1:2hz: 205 frames copied
lamp_1hz: 208 frames copied
lamp_2hz: 220 frames copied
longus_1:2hz: 193 frames copied
longus_1hz: 205 frames copied
longus_2hz: 206 frames copied
nonstop_1:2hz: 195 frames copied
nonstop_1hz: 218 frames copied
nonstop_2hz: 210 frames copied
repeater_1:2hz: 207 frames copied
repeater_1hz: 205 frames copied
repeater_2hz: 206 frames copied
rexona_1:2hz: 198 frames copied
rexona_1hz: 224 frames copied
rexona_2hz: 222 frames copied
soap_1:2hz: 

In [11]:
with open(os.path.join(base, 'list.txt'), 'w') as f:
    for seq_name in sequences:
        f.write(seq_name + '\n')

print(f"list.txt created with {len(sequences)} sequences")

list.txt created with 39 sequences


In [36]:
!python tracking/test.py hdetrack hdetrack_eventvot --dataset eventvot --threads 1 --num_gpus 1

2026-04-26 21:08:11.765234: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777237691.799441   79012 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777237691.811503   79012 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777237691.840456   79012 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777237691.840486   79012 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777237691.840493   79012 computation_placer.cc:177] computation placer alr

In [37]:
rows = []
for seq in sequences:
    gt = f'/content/EventVOT_Benchmark/HDETrack/data/EventVOT/test/{seq}/groundtruth.txt'
    pr = f'/content/EventVOT_Benchmark/HDETrack/output/test/tracking_results/hdetrack/hdetrack_eventvot/{seq}.txt'

    gt_data = np.loadtxt(gt, delimiter=',')
    pr_data = np.loadtxt(pr)
    pr, sr = calc_metrics(gt_data, pr_data)

    rows.append({
        "sequence":     seq,
        "PR (HDE)":     pr,
        "SR@0.5 (HDE)": sr,
    })

df = pd.DataFrame(rows)

# Print sequences with 2 decimals
print(df.to_string(index=False, float_format=lambda x: f"{x:.2f}"))

# Print mean separately with 3 decimals
print("-" * 40)
print(f"{'MEAN':<25} {df['PR (HDE)'].mean():.3f}   {df['SR@0.5 (HDE)'].mean():.3f}")

      sequence  PR (HDE)  SR@0.5 (HDE)
    ball_1:2hz      0.45          0.12
      ball_1hz      0.65          0.19
      ball_2hz      0.43          0.07
  banana_1:2hz      0.65          0.40
    banana_1hz      0.49          0.25
    banana_2hz      0.23          0.12
 cleaner_1:2hz      0.87          0.60
   cleaner_1hz      0.38          0.22
   cleaner_2hz      0.43          0.21
      d3_1:2hz      0.45          0.31
        d3_1hz      0.64          0.40
        d3_2hz      0.44          0.32
     key_1:2hz      0.76          0.22
       key_1hz      0.21          0.05
       key_2hz      0.13          0.00
   knife_1:2hz      0.21          0.09
     knife_1hz      0.23          0.04
     knife_2hz      0.15          0.06
    lamp_1:2hz      0.67          0.43
      lamp_1hz      0.50          0.23
      lamp_2hz      0.44          0.10
  longus_1:2hz      0.45          0.10
    longus_1hz      0.46          0.16
    longus_2hz      0.12          0.01
 nonstop_1:2hz      0.69 

## fine_tune

In [23]:
file_path = '/content/EventVOT_Benchmark/HDETrack/lib/train/dataset/__init__.py'

with open(file_path, 'r') as f:
    content = f.read()

old = "from .eventvot import EventVOT"
new = "from .eventvot import EventVOT\nfrom .illumination import Illumination"

content = content.replace(old, new)

with open(file_path, 'w') as f:
    f.write(content)

print("Done!")
print(open(file_path).read())

Done!
from .coesot import Coesot
from .fe108 import Fe108
from .visevent import VisEvent
from .eventvot import EventVOT
from .illumination import Illumination


In [24]:
file_path = '/content/EventVOT_Benchmark/HDETrack/lib/train/admin/local.py'

with open(file_path, 'r') as f:
    content = f.read()

old = "self.eventvot_dir"
new = "self.eventvot_dir"

# Add illumination_dir after the last self.xxx line
content = content.rstrip() + "\n        self.illumination_dir = '/content/EventVOT_Benchmark/HDETrack/data/Illumination/train'\n"

with open(file_path, 'w') as f:
    f.write(content)

print("Done!")

Done!


In [25]:
%cd /content/EventVOT_Benchmark/HDETrack

!python tracking/create_default_local_file.py \
    --workspace_dir . \
    --data_dir ./data \
    --save_dir ./output

/content/EventVOT_Benchmark/HDETrack
2026-04-27 18:34:11.460350: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777314851.480781   31751 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777314851.489123   31751 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777314851.508066   31751 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777314851.508107   31751 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777314851.508111   31751 computation_

In [26]:
file_path = '/content/EventVOT_Benchmark/HDETrack/lib/train/admin/local.py'
with open(file_path, 'r') as f:
    content = f.read()

if 'illumination_dir' not in content:
    content = content.rstrip() + "\n        self.illumination_dir = '/content/EventVOT_Benchmark/HDETrack/data/Illumination/train'\n"
    with open(file_path, 'w') as f:
        f.write(content)
    print("illumination_dir added")
else:
    print("illumination_dir already present")

print(open(file_path).read())

illumination_dir added
class EnvironmentSettings:
    def __init__(self):
        self.workspace_dir = '/content/EventVOT_Benchmark/HDETrack'    # Base directory for saving network checkpoints.
        self.tensorboard_dir = '/content/EventVOT_Benchmark/HDETrack/tensorboard'    # Directory for tensorboard files.
        self.pretrained_networks = '/content/EventVOT_Benchmark/HDETrack/pretrained_networks'
        self.coesot_dir = '/content/EventVOT_Benchmark/HDETrack/data/COESOT/train'
        self.coesot_val_dir = '/content/EventVOT_Benchmark/HDETrack/data/COESOT/test'
        self.fe108_dir = '/content/EventVOT_Benchmark/HDETrack/data/FE108/train'
        self.fe108_val_dir = '/content/EventVOT_Benchmark/HDETrack/data/FE108/test'
        self.visevent_dir = '/content/EventVOT_Benchmark/HDETrack/data/VisEvent/train'
        self.visevent_val_dir = '/content/EventVOT_Benchmark/HDETrack/data/VisEvent/test'
        self.eventvot_dir = '/content/EventVOT_Benchmark/HDETrack/data/EventVOT/E

In [27]:
import shutil

ill = '/content/drive/MyDrive/Diploma_project/HDETrack/illumination.py'
yaml = '/content/drive/MyDrive/Diploma_project/HDETrack/hdetrack_illumination.yaml'

shutil.copy(ill, '/content/EventVOT_Benchmark/HDETrack/lib/train/dataset/illumination.py')
print("illumination.py copied")

shutil.copy(yaml, '/content/EventVOT_Benchmark/HDETrack/experiments/hdetrack/hdetrack_illumination.yaml')
print("hdetrack_illumination.yaml copied")

illumination.py copied
hdetrack_illumination.yaml copied


In [13]:
import os
import shutil
from tqdm.notebook import tqdm

base_drive = '/content/drive/MyDrive/Diploma_project/Datasets/Illumination_custom/seya/clean'
base_dst = '/content/EventVOT_Benchmark/HDETrack/data/Illumination/train'
sequences = ['bracket', 'canada', 'clamp', 'lighter', 'ruler', 'spinner', 'tag', 'traumel']

for seq in tqdm(sequences):
    gt = f'{base_drive}/{seq}/groundtruth_rect.txt'
    for i in range(1, 11):
        dst_img = f'{base_dst}/{seq}_aug{i}/img'
        os.makedirs(dst_img, exist_ok=True)

        shutil.copy(gt, f'{base_dst}/{seq}_aug{i}/groundtruth.txt')

        src_dvs = f'{base_drive}/{seq}/augmented/dvs_{i}'
        for fname in sorted(os.listdir(src_dvs)):
            if fname.endswith('.png') or fname.endswith('.bmp'):
                shutil.copy(os.path.join(src_dvs, fname), os.path.join(dst_img, fname))

        print(f'Done: {seq}_aug{i}')

  0%|          | 0/8 [00:00<?, ?it/s]

Done: bracket_aug1
Done: bracket_aug2
Done: bracket_aug3
Done: bracket_aug4
Done: bracket_aug5
Done: bracket_aug6
Done: bracket_aug7
Done: bracket_aug8
Done: bracket_aug9
Done: bracket_aug10
Done: canada_aug1
Done: canada_aug2
Done: canada_aug3
Done: canada_aug4
Done: canada_aug5
Done: canada_aug6
Done: canada_aug7
Done: canada_aug8
Done: canada_aug9
Done: canada_aug10
Done: clamp_aug1
Done: clamp_aug2
Done: clamp_aug3
Done: clamp_aug4
Done: clamp_aug5
Done: clamp_aug6
Done: clamp_aug7
Done: clamp_aug8
Done: clamp_aug9
Done: clamp_aug10
Done: lighter_aug1
Done: lighter_aug2
Done: lighter_aug3
Done: lighter_aug4
Done: lighter_aug5
Done: lighter_aug6
Done: lighter_aug7
Done: lighter_aug8
Done: lighter_aug9
Done: lighter_aug10
Done: ruler_aug1
Done: ruler_aug2
Done: ruler_aug3
Done: ruler_aug4
Done: ruler_aug5
Done: ruler_aug6
Done: ruler_aug7
Done: ruler_aug8
Done: ruler_aug9
Done: ruler_aug10
Done: spinner_aug1
Done: spinner_aug2
Done: spinner_aug3
Done: spinner_aug4
Done: spinner_aug5


In [14]:
import os
from pathlib import Path

base = '/content/EventVOT_Benchmark/HDETrack'
train_dir = Path(f'{base}/data/Illumination/train')

all_seqs = sorted([s for s in os.listdir(train_dir) if os.path.isdir(train_dir / s)])
print(f"Found {len(all_seqs)} sequences")

val_seqs = [s for s in all_seqs if s.endswith('_aug1')]
train_seqs = [s for s in all_seqs if s not in val_seqs]

# list.txt - all sequences
(train_dir / 'list.txt').write_text('\n'.join(all_seqs) + '\n')

# train.txt and val.txt - indices
all_seqs_list = list(all_seqs)
train_indices = [str(all_seqs_list.index(s)) for s in train_seqs]
val_indices = [str(all_seqs_list.index(s)) for s in val_seqs]

(train_dir / 'train.txt').write_text('\n'.join(train_indices) + '\n')
(train_dir / 'val.txt').write_text('\n'.join(val_indices) + '\n')

print(f"Train: {len(train_seqs)} sequences")
print(f"Val: {len(val_seqs)} sequences")
print("\nVal sequences:", val_seqs)

Found 80 sequences
Train: 72 sequences
Val: 8 sequences

Val sequences: ['bracket_aug1', 'canada_aug1', 'clamp_aug1', 'lighter_aug1', 'ruler_aug1', 'spinner_aug1', 'tag_aug1', 'traumel_aug1']


In [15]:
import os
import shutil
import torch

# Create checkpoint dir for new experiment
ckpt_dir = '/content/EventVOT_Benchmark/HDETrack/output/checkpoints/train/hdetrack/hdetrack_illumination'
os.makedirs(ckpt_dir, exist_ok=True)

# Copy pretrained checkpoint
src = '/content/EventVOT_Benchmark/HDETrack/output/checkpoints/train/hdetrack/hdetrack_eventvot/HDETrack_S_ep0050.pth.tar'
dst = f'{ckpt_dir}/HDETrack_S_ep0050.pth.tar'
shutil.copy(src, dst)
print("Checkpoint copied")

# Reset epoch to 0
ckpt = torch.load(dst, map_location='cpu', weights_only=False)
ckpt['epoch'] = 0
torch.save(ckpt, dst)
print("Epoch reset to 0")

Checkpoint copied
Epoch reset to 0


In [17]:
file_path = '/content/EventVOT_Benchmark/HDETrack/lib/train/base_functions.py'

with open(file_path, 'r') as f:
    content = f.read()

# Add Illumination to whitelist
old = '"COESOT", "COESOT_VAL", "FE108", "FE108_VAL", "VisEvent", "VisEvent_VAL","EventVOT"]'
new = '"COESOT", "COESOT_VAL", "FE108", "FE108_VAL", "VisEvent", "VisEvent_VAL","EventVOT", "Illumination"]'
content = content.replace(old, new)

# Add Illumination dataset handler
old = '        if name == "EventVOT":\n            datasets.append(EventVOT(settings.env.eventvot_dir, split=\'train\', image_loader=image_loader))\n'
new = '        if name == "EventVOT":\n            datasets.append(EventVOT(settings.env.eventvot_dir, split=\'train\', image_loader=image_loader))\n        if name == "Illumination":\n            datasets.append(Illumination(settings.env.illumination_dir, split=\'train\', image_loader=image_loader))\n'
content = content.replace(old, new)

with open(file_path, 'w') as f:
    f.write(content)

print("Done!")

Done!


In [21]:
file_path = '/content/EventVOT_Benchmark/HDETrack/lib/train/base_functions.py'

with open(file_path, 'r') as f:
    content = f.read()

old = 'from lib.train.dataset import EventVOT # COESOT, FE108, VisEvent'
new = 'from lib.train.dataset import EventVOT, Illumination # COESOT, FE108, VisEvent'
content = content.replace(old, new)

with open(file_path, 'w') as f:
    f.write(content)

print("Done!")
!grep -n "from lib.train.dataset" /content/EventVOT_Benchmark/HDETrack/lib/train/base_functions.py

Done!
4:from lib.train.dataset import EventVOT, Illumination # COESOT, FE108, VisEvent


In [28]:
file_path = '/content/EventVOT_Benchmark/HDETrack/lib/models/hdetrack/vit_ce.py'

with open(file_path, 'r') as f:
    content = f.read()

content = content.replace(
    'checkpoint = torch.load(pretrained, map_location="cpu")',
    'checkpoint = torch.load(pretrained, map_location="cpu", weights_only=False)'
)

with open(file_path, 'w') as f:
    f.write(content)

print("Done!")

Done!


In [29]:
file_path = '/content/EventVOT_Benchmark/HDETrack/lib/models/hdetrack/hdetrack.py'

with open(file_path, 'r') as f:
    content = f.read()

content = content.replace(
    'cheakpoint=torch.load(os.path.join(pretrained_path, cfg.MODEL.PRETRAIN_FILE_T))',
    'cheakpoint=torch.load(os.path.join(pretrained_path, cfg.MODEL.PRETRAIN_FILE_T), weights_only=False)'
)

with open(file_path, 'w') as f:
    f.write(content)

print("Done!")

Done!


In [30]:
file_path = '/content/EventVOT_Benchmark/HDETrack/lib/train/trainers/base_trainer.py'

with open(file_path, 'r') as f:
    content = f.read()

content = content.replace(
    "checkpoint_dict = torch.load(checkpoint_path, map_location='cpu')",
    "checkpoint_dict = torch.load(checkpoint_path, map_location='cpu', weights_only=False)"
)

with open(file_path, 'w') as f:
    f.write(content)

print("Done!")

Done!


In [29]:
file_path = '/content/EventVOT_Benchmark/HDETrack/lib/train/data/loader.py'

with open(file_path, 'r') as f:
    content = f.read()

content = content.replace(
    'elif isinstance(batch[0], collections.Mapping):',
    'elif isinstance(batch[0], collections.abc.Mapping):'
)

with open(file_path, 'w') as f:
    f.write(content)

print("Done!")

Done!


In [31]:
file_path = '/content/EventVOT_Benchmark/HDETrack/lib/train/dataset/illumination.py'

with open(file_path, 'r') as f:
    content = f.read()

content = content.replace(
    "frame_list = [self._get_frame(seq_path, f_id) for f_id in frame_ids]",
    "frame_list = [self._get_frame(seq_path, f_id + 1) for f_id in frame_ids]"
)

with open(file_path, 'w') as f:
    f.write(content)

print("Done!")

Done!


In [32]:
file_path = '/content/EventVOT_Benchmark/HDETrack/lib/train/data/loader.py'

with open(file_path, 'r') as f:
    content = f.read()

content = content.replace(
    'elif isinstance(batch[0], collections.Sequence):',
    'elif isinstance(batch[0], collections.abc.Sequence):'
)

with open(file_path, 'w') as f:
    f.write(content)

print("Done!")

Done!


In [35]:
file_path = '/content/EventVOT_Benchmark/HDETrack/lib/train/data/loader.py'

with open(file_path, 'r') as f:
    content = f.read()

# Fix the actual cause - use untyped_storage instead of deprecated storage()
content = content.replace(
    'storage = batch[0].storage()._new_shared(numel)',
    'storage = batch[0].untyped_storage()._new_shared(numel * batch[0].element_size())'
)

# Fix the out= argument issue - remove out= to avoid resize warnings
content = content.replace(
    'return torch.stack(batch, 1, out=out)',
    'return torch.stack(batch, 1)'
)

with open(file_path, 'w') as f:
    f.write(content)

print("Done!")

Done!


In [36]:
import warnings
warnings.filterwarnings("ignore")

%cd /content/EventVOT_Benchmark/HDETrack
!python tracking/train.py --script hdetrack --config hdetrack_illumination --save_dir ./output --mode single --nproc_per_node 1 --use_wandb 0

/content/EventVOT_Benchmark/HDETrack
2026-04-27 09:34:29.906284: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777282469.928477   57279 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777282469.935890   57279 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777282469.954604   57279 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777282469.954630   57279 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777282469.954634   57279 computation_

In [37]:
src = '/content/EventVOT_Benchmark/HDETrack/output/checkpoints/train/hdetrack/hdetrack_illumination'
dst = '/content/drive/MyDrive/Diploma_project/HDETrack/checkpoints'

os.makedirs(dst, exist_ok=True)

for f in os.listdir(src):
    if f != 'HDETrack_ep0050.pth.tar':
        shutil.copy(os.path.join(src, f), os.path.join(dst, f))
        print(f'Copied: {f}')

Copied: HDETrack_S_ep0028.pth.tar
Copied: HDETrack_S_ep0050.pth.tar
Copied: HDETrack_S_ep0030.pth.tar
Copied: HDETrack_S_ep0010.pth.tar
Copied: HDETrack_S_ep0020.pth.tar
Copied: HDETrack_S_ep0029.pth.tar


### Inference

In [ ]:
!rm -rf '/content/SDTrack/SDTrack-Event/output'

In [40]:
sequences = [
    s for s in sorted(os.listdir('/content/drive/MyDrive/Diploma_project/Datasets/Illumination_custom/seya/flicker'))[1:]
    if not s.endswith('.zip')
    and not s.startswith('pen')
    and not s.startswith('spinner')
    and not s.endswith('4hz')
    and not s.endswith('8hz')
]

In [42]:
src_root = '/content/drive/MyDrive/Diploma_project/Datasets/Illumination_custom/seya/flicker'
base = '/content/EventVOT_Benchmark/HDETrack/data/EventVOT/test'


# Copy each sequence
for seq_name in sequences:
    dvs = os.path.join(src_root, seq_name, 'dvs')
    gt = os.path.join(src_root, seq_name, 'groundtruth_rect.txt')
    img_dst = os.path.join(base, seq_name, 'img')

    os.makedirs(img_dst, exist_ok=True)

    # Copy frames
    frames = [f for f in sorted(os.listdir(dvs)) if f.endswith('.png') or f.endswith('.bmp')]
    for fname in frames:
        shutil.copy(os.path.join(dvs, fname), os.path.join(img_dst, fname))

    # Copy groundtruth
    shutil.copy(gt, os.path.join(base, seq_name, 'groundtruth.txt'))

    print(f"{seq_name}: {len(frames)} frames copied")

print("\nDone!")

ball_1:2hz: 214 frames copied
ball_1hz: 202 frames copied
ball_2hz: 204 frames copied
banana_1:2hz: 246 frames copied
banana_1hz: 226 frames copied
banana_2hz: 213 frames copied
cleaner_1:2hz: 206 frames copied
cleaner_1hz: 207 frames copied
cleaner_2hz: 191 frames copied
d3_1:2hz: 209 frames copied
d3_1hz: 215 frames copied
d3_2hz: 209 frames copied
key_1:2hz: 217 frames copied
key_1hz: 221 frames copied
key_2hz: 207 frames copied
knife_1:2hz: 191 frames copied
knife_1hz: 205 frames copied
knife_2hz: 209 frames copied
lamp_1:2hz: 205 frames copied
lamp_1hz: 208 frames copied
lamp_2hz: 220 frames copied
longus_1:2hz: 193 frames copied
longus_1hz: 205 frames copied
longus_2hz: 206 frames copied
nonstop_1:2hz: 195 frames copied
nonstop_1hz: 218 frames copied
nonstop_2hz: 210 frames copied
repeater_1:2hz: 207 frames copied
repeater_1hz: 205 frames copied
repeater_2hz: 206 frames copied
rexona_1:2hz: 198 frames copied
rexona_1hz: 224 frames copied
rexona_2hz: 222 frames copied
soap_1:2hz: 

In [44]:
with open(os.path.join(base, 'list.txt'), 'w') as f:
    for seq_name in sequences:
        f.write(seq_name + '\n')

print(f"list.txt created with {len(sequences)} sequences")

list.txt created with 39 sequences


In [45]:
%cd /content/EventVOT_Benchmark/HDETrack

!python tracking/test.py hdetrack hdetrack_illumination \
    --dataset eventvot \
    --threads 1 \
    --num_gpus 1

/content/EventVOT_Benchmark/HDETrack
2026-04-27 11:39:07.146992: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777289947.168892  106995 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777289947.176193  106995 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777289947.194990  106995 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777289947.195015  106995 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777289947.195018  106995 computation_

In [49]:
def calc_metrics(gt, pr):
        min_len = min(len(gt), len(pr))
        gt = gt[:min_len]
        pr = pr[:min_len]

        gt_centers = gt[:, :2] + gt[:, 2:4] / 2
        pr_centers = pr[:, :2] + pr[:, 2:4] / 2
        distances  = np.sqrt(np.sum((gt_centers - pr_centers) ** 2, axis=1))
        precision  = np.mean(distances <= 20)

        ix1 = np.maximum(gt[:, 0], pr[:, 0])
        iy1 = np.maximum(gt[:, 1], pr[:, 1])
        ix2 = np.minimum(gt[:, 0] + gt[:, 2], pr[:, 0] + pr[:, 2])
        iy2 = np.minimum(gt[:, 1] + gt[:, 3], pr[:, 1] + pr[:, 3])

        inter_area = np.maximum(0, ix2 - ix1) * np.maximum(0, iy2 - iy1)
        union_area = gt[:, 2] * gt[:, 3] + pr[:, 2] * pr[:, 3] - inter_area
        iou        = inter_area / (union_area + 1e-10)
        success    = np.mean(iou > 0.5)

        return precision, success

In [50]:
rows = []
for seq in sequences:
    gt = f'/content/EventVOT_Benchmark/HDETrack/data/EventVOT/test/{seq}/groundtruth.txt'
    pr = f'/content/EventVOT_Benchmark/HDETrack/output/test/tracking_results/hdetrack/hdetrack_illumination/{seq}.txt'

    gt_data = np.loadtxt(gt, delimiter=',')
    pr_data = np.loadtxt(pr)
    pr, sr = calc_metrics(gt_data, pr_data)

    rows.append({
        "sequence":     seq,
        "PR (HDE)":     pr,
        "SR@0.5 (HDE)": sr,
    })

df = pd.DataFrame(rows)

# Print sequences with 2 decimals
print(df.to_string(index=False, float_format=lambda x: f"{x:.2f}"))

# Print mean separately with 3 decimals
print("-" * 40)
print(f"{'MEAN':<25} {df['PR (HDE)'].mean():.3f}   {df['SR@0.5 (HDE)'].mean():.3f}")

      sequence  PR (HDE)  SR@0.5 (HDE)
    ball_1:2hz      0.33          0.11
      ball_1hz      0.69          0.21
      ball_2hz      0.29          0.04
  banana_1:2hz      0.67          0.48
    banana_1hz      0.42          0.25
    banana_2hz      0.28          0.06
 cleaner_1:2hz      0.96          0.72
   cleaner_1hz      0.71          0.42
   cleaner_2hz      0.59          0.25
      d3_1:2hz      0.89          0.75
        d3_1hz      0.72          0.49
        d3_2hz      0.83          0.53
     key_1:2hz      0.92          0.41
       key_1hz      0.89          0.27
       key_2hz      0.42          0.03
   knife_1:2hz      0.52          0.26
     knife_1hz      0.72          0.29
     knife_2hz      0.12          0.01
    lamp_1:2hz      0.61          0.50
      lamp_1hz      0.62          0.41
      lamp_2hz      0.50          0.21
  longus_1:2hz      0.68          0.36
    longus_1hz      0.47          0.18
    longus_2hz      0.13          0.00
 nonstop_1:2hz      0.71 

# Redetection

In [18]:
!rm -rf '/content/EventVOT_Benchmark/HDETrack/output/test'

!python tracking/test.py hdetrack hdetrack_eventvot --dataset eventvot --threads 1 --num_gpus 1

2026-04-27 18:09:14.046799: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777313354.067873   25193 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777313354.075646   25193 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777313354.093983   25193 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777313354.094037   25193 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777313354.094043   25193 computation_placer.cc:177] computation placer alr

In [19]:
def calc_metrics(gt, pr):
        min_len = min(len(gt), len(pr))
        gt = gt[:min_len]
        pr = pr[:min_len]

        gt_centers = gt[:, :2] + gt[:, 2:4] / 2
        pr_centers = pr[:, :2] + pr[:, 2:4] / 2
        distances  = np.sqrt(np.sum((gt_centers - pr_centers) ** 2, axis=1))
        precision  = np.mean(distances <= 20)

        ix1 = np.maximum(gt[:, 0], pr[:, 0])
        iy1 = np.maximum(gt[:, 1], pr[:, 1])
        ix2 = np.minimum(gt[:, 0] + gt[:, 2], pr[:, 0] + pr[:, 2])
        iy2 = np.minimum(gt[:, 1] + gt[:, 3], pr[:, 1] + pr[:, 3])

        inter_area = np.maximum(0, ix2 - ix1) * np.maximum(0, iy2 - iy1)
        union_area = gt[:, 2] * gt[:, 3] + pr[:, 2] * pr[:, 3] - inter_area
        iou        = inter_area / (union_area + 1e-10)
        success    = np.mean(iou > 0.5)

        return precision, success

In [20]:
rows = []
for seq in sequences:
    gt = f'/content/EventVOT_Benchmark/HDETrack/data/EventVOT/test/{seq}/groundtruth.txt'
    pr = f'/content/EventVOT_Benchmark/HDETrack/output/test/tracking_results/hdetrack/hdetrack_eventvot/{seq}.txt'

    gt_data = np.loadtxt(gt, delimiter=',')
    pr_data = np.loadtxt(pr)
    pr, sr = calc_metrics(gt_data, pr_data)

    rows.append({
        "sequence":     seq,
        "PR (HDE)":     pr,
        "SR@0.5 (HDE)": sr,
    })

df = pd.DataFrame(rows)

# Print sequences with 2 decimals
print(df.to_string(index=False, float_format=lambda x: f"{x:.2f}"))

# Print mean separately with 3 decimals
print("-" * 40)
print(f"{'MEAN':<25} {df['PR (HDE)'].mean():.3f}   {df['SR@0.5 (HDE)'].mean():.3f}")

      sequence  PR (HDE)  SR@0.5 (HDE)
    ball_1:2hz      0.56          0.13
      ball_1hz      0.45          0.10
      ball_2hz      0.49          0.09
  banana_1:2hz      0.66          0.42
    banana_1hz      0.45          0.22
    banana_2hz      0.38          0.18
 cleaner_1:2hz      0.82          0.54
   cleaner_1hz      0.72          0.40
   cleaner_2hz      0.51          0.24
      d3_1:2hz      0.63          0.51
        d3_1hz      0.58          0.38
        d3_2hz      0.61          0.38
     key_1:2hz      0.77          0.20
       key_1hz      0.49          0.10
       key_2hz      0.35          0.00
   knife_1:2hz      0.50          0.27
     knife_1hz      0.55          0.20
     knife_2hz      0.29          0.08
    lamp_1:2hz      0.68          0.44
      lamp_1hz      0.56          0.24
      lamp_2hz      0.57          0.15
  longus_1:2hz      0.61          0.24
    longus_1hz      0.55          0.19
    longus_2hz      0.36          0.03
 nonstop_1:2hz      0.63 

# Redetect + fine_tune

In [33]:
!rm -rf '/content/EventVOT_Benchmark/HDETrack/output'

In [34]:
!rm -rf '/content/EventVOT_Benchmark/HDETrack/output'

hde = '/content/drive/MyDrive/Diploma_project/HDETrack/checkpoints/HDETrack_S_ep0030.pth.tar'
checkpoint_dir = '/content/EventVOT_Benchmark/HDETrack/output/checkpoints/train/hdetrack/hdetrack_illumination'

os.makedirs(checkpoint_dir, exist_ok=True)
shutil.copy(hde, os.path.join(checkpoint_dir, 'HDETrack_S_ep0030.pth.tar'))

'/content/EventVOT_Benchmark/HDETrack/output/checkpoints/train/hdetrack/hdetrack_illumination/HDETrack_S_ep0030.pth.tar'

In [35]:
%cd /content/EventVOT_Benchmark/HDETrack

!python tracking/test.py hdetrack hdetrack_illumination \
    --dataset eventvot \
    --threads 1 \
    --num_gpus 1

/content/EventVOT_Benchmark/HDETrack
2026-04-27 18:45:03.121515: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777315503.141498   34541 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777315503.148234   34541 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777315503.165772   34541 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777315503.165817   34541 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777315503.165823   34541 computation_

In [36]:
rows = []
for seq in sequences:
    gt = f'/content/EventVOT_Benchmark/HDETrack/data/EventVOT/test/{seq}/groundtruth.txt'
    pr = f'/content/EventVOT_Benchmark/HDETrack/output/test/tracking_results/hdetrack/hdetrack_illumination/{seq}.txt'

    gt_data = np.loadtxt(gt, delimiter=',')
    pr_data = np.loadtxt(pr)
    pr, sr = calc_metrics(gt_data, pr_data)

    rows.append({
        "sequence":     seq,
        "PR (HDE)":     pr,
        "SR@0.5 (HDE)": sr,
    })

df = pd.DataFrame(rows)

# Print sequences with 2 decimals
print(df.to_string(index=False, float_format=lambda x: f"{x:.2f}"))

# Print mean separately with 3 decimals
print("-" * 40)
print(f"{'MEAN':<25} {df['PR (HDE)'].mean():.3f}   {df['SR@0.5 (HDE)'].mean():.3f}")

      sequence  PR (HDE)  SR@0.5 (HDE)
    ball_1:2hz      0.36          0.09
      ball_1hz      0.65          0.19
      ball_2hz      0.31          0.07
  banana_1:2hz      0.58          0.42
    banana_1hz      0.42          0.27
    banana_2hz      0.38          0.16
 cleaner_1:2hz      0.94          0.71
   cleaner_1hz      0.74          0.42
   cleaner_2hz      0.66          0.28
      d3_1:2hz      0.74          0.61
        d3_1hz      0.63          0.38
        d3_2hz      0.74          0.44
     key_1:2hz      0.90          0.39
       key_1hz      0.57          0.17
       key_2hz      0.46          0.03
   knife_1:2hz      0.58          0.27
     knife_1hz      0.61          0.20
     knife_2hz      0.19          0.01
    lamp_1:2hz      0.61          0.48
      lamp_1hz      0.57          0.36
      lamp_2hz      0.59          0.20
  longus_1:2hz      0.65          0.33
    longus_1hz      0.48          0.16
    longus_2hz      0.29          0.01
 nonstop_1:2hz      0.68 

# Kalman

In [39]:
!rm -rf '/content/EventVOT_Benchmark/HDETrack/output'

In [40]:
# Source file paths
hde = '/content/drive/MyDrive/Diploma_project/HDETrack/HDETrack_S_ep0050.pth.tar'

checkpoint_dir = '/content/EventVOT_Benchmark/HDETrack/output/checkpoints/train/hdetrack/hdetrack_eventvot'

os.makedirs(checkpoint_dir, exist_ok=True)

# HDETrack_S (student model, for inference) → output/checkpoints/.../
shutil.copy(hde, os.path.join(checkpoint_dir, 'HDETrack_S_ep0050.pth.tar'))


'/content/EventVOT_Benchmark/HDETrack/output/checkpoints/train/hdetrack/hdetrack_eventvot/HDETrack_S_ep0050.pth.tar'

In [41]:
!rm -rf '/content/EventVOT_Benchmark/HDETrack/output/test'

!python tracking/test.py hdetrack hdetrack_eventvot --dataset eventvot --threads 1 --num_gpus 1

2026-04-27 19:14:48.723272: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777317288.760375   42288 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777317288.771872   42288 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777317288.799275   42288 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777317288.799308   42288 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777317288.799316   42288 computation_placer.cc:177] computation placer alr

In [42]:
rows = []
for seq in sequences:
    gt = f'/content/EventVOT_Benchmark/HDETrack/data/EventVOT/test/{seq}/groundtruth.txt'
    pr = f'/content/EventVOT_Benchmark/HDETrack/output/test/tracking_results/hdetrack/hdetrack_eventvot/{seq}.txt'

    gt_data = np.loadtxt(gt, delimiter=',')
    pr_data = np.loadtxt(pr)
    pr, sr = calc_metrics(gt_data, pr_data)

    rows.append({
        "sequence":     seq,
        "PR (HDE)":     pr,
        "SR@0.5 (HDE)": sr,
    })

df = pd.DataFrame(rows)

# Print sequences with 2 decimals
print(df.to_string(index=False, float_format=lambda x: f"{x:.2f}"))

# Print mean separately with 3 decimals
print("-" * 40)
print(f"{'MEAN':<25} {df['PR (HDE)'].mean():.3f}   {df['SR@0.5 (HDE)'].mean():.3f}")

      sequence  PR (HDE)  SR@0.5 (HDE)
    ball_1:2hz      0.45          0.12
      ball_1hz      0.68          0.18
      ball_2hz      0.42          0.06
  banana_1:2hz      0.65          0.40
    banana_1hz      0.50          0.24
    banana_2hz      0.27          0.13
 cleaner_1:2hz      0.90          0.62
   cleaner_1hz      0.43          0.22
   cleaner_2hz      0.45          0.16
      d3_1:2hz      0.45          0.32
        d3_1hz      0.65          0.37
        d3_2hz      0.49          0.31
     key_1:2hz      0.78          0.20
       key_1hz      0.22          0.05
       key_2hz      0.12          0.00
   knife_1:2hz      0.19          0.08
     knife_1hz      0.22          0.03
     knife_2hz      0.16          0.04
    lamp_1:2hz      0.68          0.43
      lamp_1hz      0.54          0.23
      lamp_2hz      0.42          0.10
  longus_1:2hz      0.46          0.10
    longus_1hz      0.50          0.14
    longus_2hz      0.14          0.01
 nonstop_1:2hz      0.69 

# Kalman fine_tune

In [37]:
!rm -rf '/content/EventVOT_Benchmark/HDETrack/output/test'

%cd /content/EventVOT_Benchmark/HDETrack

!python tracking/test.py hdetrack hdetrack_illumination \
    --dataset eventvot \
    --threads 1 \
    --num_gpus 1

/content/EventVOT_Benchmark/HDETrack
2026-04-27 19:04:20.499396: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777316660.519806   39550 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777316660.526803   39550 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777316660.544876   39550 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777316660.544920   39550 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777316660.544925   39550 computation_

In [38]:
rows = []
for seq in sequences:
    gt = f'/content/EventVOT_Benchmark/HDETrack/data/EventVOT/test/{seq}/groundtruth.txt'
    pr = f'/content/EventVOT_Benchmark/HDETrack/output/test/tracking_results/hdetrack/hdetrack_illumination/{seq}.txt'

    gt_data = np.loadtxt(gt, delimiter=',')
    pr_data = np.loadtxt(pr)
    pr, sr = calc_metrics(gt_data, pr_data)

    rows.append({
        "sequence":     seq,
        "PR (HDE)":     pr,
        "SR@0.5 (HDE)": sr,
    })

df = pd.DataFrame(rows)

# Print sequences with 2 decimals
print(df.to_string(index=False, float_format=lambda x: f"{x:.2f}"))

# Print mean separately with 3 decimals
print("-" * 40)
print(f"{'MEAN':<25} {df['PR (HDE)'].mean():.3f}   {df['SR@0.5 (HDE)'].mean():.3f}")

      sequence  PR (HDE)  SR@0.5 (HDE)
    ball_1:2hz      0.36          0.14
      ball_1hz      0.52          0.13
      ball_2hz      0.28          0.05
  banana_1:2hz      0.66          0.48
    banana_1hz      0.57          0.35
    banana_2hz      0.27          0.07
 cleaner_1:2hz      0.93          0.68
   cleaner_1hz      0.69          0.42
   cleaner_2hz      0.41          0.15
      d3_1:2hz      0.87          0.75
        d3_1hz      0.75          0.46
        d3_2hz      0.64          0.41
     key_1:2hz      0.93          0.38
       key_1hz      0.90          0.24
       key_2hz      0.53          0.05
   knife_1:2hz      0.50          0.27
     knife_1hz      0.66          0.27
     knife_2hz      0.10          0.01
    lamp_1:2hz      0.61          0.50
      lamp_1hz      0.65          0.45
      lamp_2hz      0.51          0.19
  longus_1:2hz      0.66          0.37
    longus_1hz      0.53          0.25
    longus_2hz      0.13          0.00
 nonstop_1:2hz      0.67 

# Fine_tune + Kalman + Redetection

In [43]:
!rm -rf '/content/EventVOT_Benchmark/HDETrack/output'

hde = '/content/drive/MyDrive/Diploma_project/HDETrack/checkpoints/HDETrack_S_ep0030.pth.tar'
checkpoint_dir = '/content/EventVOT_Benchmark/HDETrack/output/checkpoints/train/hdetrack/hdetrack_illumination'

os.makedirs(checkpoint_dir, exist_ok=True)
shutil.copy(hde, os.path.join(checkpoint_dir, 'HDETrack_S_ep0030.pth.tar'))

'/content/EventVOT_Benchmark/HDETrack/output/checkpoints/train/hdetrack/hdetrack_illumination/HDETrack_S_ep0030.pth.tar'

In [44]:
!rm -rf '/content/EventVOT_Benchmark/HDETrack/output/test'

%cd /content/EventVOT_Benchmark/HDETrack

!python tracking/test.py hdetrack hdetrack_illumination \
    --dataset eventvot \
    --threads 1 \
    --num_gpus 1

/content/EventVOT_Benchmark/HDETrack
2026-04-27 19:26:23.916513: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777317983.936761   45314 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777317983.944063   45314 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777317983.961596   45314 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777317983.961640   45314 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777317983.961645   45314 computation_

In [45]:
rows = []
for seq in sequences:
    gt = f'/content/EventVOT_Benchmark/HDETrack/data/EventVOT/test/{seq}/groundtruth.txt'
    pr = f'/content/EventVOT_Benchmark/HDETrack/output/test/tracking_results/hdetrack/hdetrack_illumination/{seq}.txt'

    gt_data = np.loadtxt(gt, delimiter=',')
    pr_data = np.loadtxt(pr)
    pr, sr = calc_metrics(gt_data, pr_data)

    rows.append({
        "sequence":     seq,
        "PR (HDE)":     pr,
        "SR@0.5 (HDE)": sr,
    })

df = pd.DataFrame(rows)

# Print sequences with 2 decimals
print(df.to_string(index=False, float_format=lambda x: f"{x:.2f}"))

# Print mean separately with 3 decimals
print("-" * 40)
print(f"{'MEAN':<25} {df['PR (HDE)'].mean():.3f}   {df['SR@0.5 (HDE)'].mean():.3f}")

      sequence  PR (HDE)  SR@0.5 (HDE)
    ball_1:2hz      0.36          0.09
      ball_1hz      0.66          0.19
      ball_2hz      0.31          0.06
  banana_1:2hz      0.59          0.43
    banana_1hz      0.44          0.27
    banana_2hz      0.39          0.15
 cleaner_1:2hz      0.96          0.72
   cleaner_1hz      0.75          0.42
   cleaner_2hz      0.62          0.20
      d3_1:2hz      0.76          0.62
        d3_1hz      0.65          0.37
        d3_2hz      0.73          0.42
     key_1:2hz      0.89          0.37
       key_1hz      0.57          0.17
       key_2hz      0.41          0.03
   knife_1:2hz      0.57          0.29
     knife_1hz      0.61          0.20
     knife_2hz      0.18          0.03
    lamp_1:2hz      0.60          0.50
      lamp_1hz      0.60          0.37
      lamp_2hz      0.55          0.16
  longus_1:2hz      0.63          0.34
    longus_1hz      0.47          0.15
    longus_2hz      0.25          0.01
 nonstop_1:2hz      0.67 